# 한입안심 데이터 품질 감사

**TL;DR**: 공식 재수집 후 631개 메뉴·9개 브랜드를 확보했고 `(brand, menu)` 중복과 품질 오류는 0건이다. 다만 파리바게뜨 1행, 스타벅스·써브웨이의 알레르기 확인율 0%는 해소되지 않은 범위 한계다.

## Context & Methods

`data/menus.csv`와 자동 품질 보고서를 불러와 행 수, 중복, 수치 범위, 브랜드별 알레르기 확인율, 두 앱 CSV 일치 여부를 재검산한다. 이 노트북은 저장소 루트에서 실행한다.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'menus.csv').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'menus.csv'
MIRROR = ROOT / 'vercel-app' / 'public' / 'data' / 'menus.csv'
REPORT = ROOT / 'reports' / 'data-quality.json'
menus = pd.read_csv(DATA)
quality = json.loads(REPORT.read_text(encoding='utf-8'))
len(menus), menus['brand'].nunique(), quality['status']

(631, 9, 'pass')

## Data

In [2]:
coverage = (
    menus.groupby('brand')
    .agg(rows=('menu', 'size'), allergen_known=('allergen_known', 'sum'), latest_source_date=('source_date', 'max'))
    .assign(allergen_known_rate=lambda frame: (frame['allergen_known'] / frame['rows'] * 100).round(1))
    .sort_values('rows', ascending=False)
)
coverage

,rows,allergen_known,latest_source_date,allergen_known_rate
brand,,,,
버거킹,210,210,2026-08-11,100.0
스타벅스,192,0,2026-08-11,0.0
롯데리아,100,100,2026-07-16,100.0
맥도날드,39,39,2026-08-11,100.0
배스킨라빈스,31,31,2026-08-11,100.0
써브웨이,25,0,2026-08-11,0.0
KFC,17,17,2026-07-21,100.0
이디야,16,8,2026-08-11,50.0
파리바게뜨,1,1,2026-08-11,100.0


## Results

In [3]:
numeric = menus[['calories', 'protein', 'fat', 'carbs', 'sodium']].agg(['min', 'max']).T
duplicate_count = int(menus.duplicated(['brand', 'menu']).sum())
primary_hash = hashlib.sha256(DATA.read_bytes()).hexdigest()
mirror_hash = hashlib.sha256(MIRROR.read_bytes()).hexdigest()
summary = {
    'rows': len(menus),
    'brands': int(menus['brand'].nunique()),
    'duplicates': duplicate_count,
    'validator_errors': quality['summary']['errors'],
    'validator_warnings': quality['summary']['warnings'],
    'mirror_identical': primary_hash == mirror_hash,
}
summary, numeric

({'rows': 631,
  'brands': 9,
  'duplicates': 0,
  'validator_errors': 0,
  'validator_warnings': 1,
  'mirror_identical': True},
           min     max
 calories  0.0  1812.0
 protein   0.0   197.0
 fat       0.0    58.0
 carbs     0.0    99.0
 sodium    0.0  3351.0)

In [4]:
assert summary == {
    'rows': 631, 'brands': 9, 'duplicates': 0,
    'validator_errors': 0, 'validator_warnings': 1, 'mirror_identical': True,
}
assert menus['source_url'].str.startswith('https://').all()
assert set(menus['source_date_type']) <= {'official_updated', 'official_published', 'official_version', 'collected_on'}
'품질 감사 assertions 통과'

'품질 감사 assertions 통과'

## Takeaways

- 구조적 오류와 두 앱 데이터 불일치는 현재 게이트에서 차단된다.
- 파리바게뜨 1행은 공식 페이지 파싱 범위를 넓혀야 하는 우선 과제다.
- 스타벅스·써브웨이 알레르기 확인율 0%는 안전상 보수적 제외가 필요하다.
- 자동 검사는 공식 의미의 완전성을 보증하지 않으므로 브랜드별 수동 표본 대조를 이어가야 한다.